# 01 · Bronze layer check

Run top to bottom after the two Phase 3 jobs (README, Phase 3). Same kernel setup as notebook 00.

You'll see: every Bronze table and its row count, what a raw order looks like, the duplicate
orders Bronze deliberately keeps, and Delta Lake's transaction history (time travel).

## 1. Start Spark, wired for Delta Lake + MinIO

`get_spark()` lives in `src/common/spark_session.py`; every job uses the same one.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/src')  # so the notebook can import our shared code

from pyspark.sql import functions as F
from common.spark_session import get_spark, lake_path

spark = get_spark('bronze-check')
print('Spark', spark.version)

## 2. Every Bronze table and its row count

Expected: orders grows with the stream; the other 8 match the CSVs (customers 99,441, order_items 112,650, ...).

In [ ]:
tables = ['orders', 'customers', 'order_items', 'order_payments', 'order_reviews',
          'products', 'sellers', 'geolocation', 'product_category_translation']
for t in tables:
    print(f"{t:<30} {spark.read.format('delta').load(lake_path('bronze', t)).count():>10,}")

## 3. What a raw order looks like

The original fields (all strings: Bronze never casts), then the Kafka metadata that says exactly where each row came from.

In [ ]:
orders = spark.read.format('delta').load(lake_path('bronze', 'orders'))
orders.printSchema()
orders.drop('_raw_value').show(5, truncate=False)

## 4. Duplicates: kept on purpose

Each Kafka message is written once (partition + offset is unique), but the same *order* can appear
several times because the producer was run more than once. Bronze records what happened;
Phase 4 (Silver) deduplicates.

In [ ]:
print('rows                  ', orders.count())
print('unique Kafka messages ', orders.select('_kafka_partition', '_kafka_offset').distinct().count())
print('unique orders         ', orders.select('order_id').distinct().count())

(orders.groupBy('order_id').count()
       .groupBy(F.col('count').alias('copies_per_order')).count()
       .orderBy('copies_per_order').show())

## 5. Delta Lake history: every write is a versioned transaction

Each streaming micro-batch is one version. This is what the `_delta_log/` folder in MinIO records.

In [ ]:
(spark.sql(f"DESCRIBE HISTORY delta.`{lake_path('bronze', 'orders')}`")
      .select('version', 'timestamp', 'operation', 'operationMetrics.numOutputRows')
      .show(truncate=False))

## 6. Time travel: read the table as it was at version 0

In [ ]:
v0 = spark.read.format('delta').option('versionAsOf', 0).load(lake_path('bronze', 'orders'))
print('rows at version 0:', v0.count(), '| rows now:', orders.count())

## All green? Phase 3 is done.

Also open the MinIO console (http://localhost:9001) → bucket `lake` → `bronze/` to see the Parquet files and `_delta_log/` folders.

Next: Phase 4, the Silver layer (clean, cast types, dedupe, join).